# Task 7 — Anomaly Detection & Promotion Impact Analysis

1. Detect anomalous sales days using Z-score and IQR methods.
2. Quantify the causal lift from M5 events (Super Bowl, Thanksgiving, etc.)
   using interrupted time-series regression.

**Pre-requisite:** `data/processed/m5_long.parquet` exists.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from m5forecast.analysis.anomaly_impact import (
    detect_anomalies_zscore,
    batch_event_impact,
)

ROOT = Path("..")
PARQUET = ROOT / "data" / "processed" / "m5_long.parquet"
assert PARQUET.exists()

df = pd.read_parquet(PARQUET)
df["date"] = pd.to_datetime(df["date"])

## 1. Anomaly detection on store CA_1

In [ ]:
series = (
    df[df["store_id"] == "CA_1"]
    .groupby("date")["sales"]
    .sum()
    .sort_index()
)

flags = detect_anomalies_zscore(series, window=28, threshold=3.0)
anomalies = series[flags]
print(f"Detected {len(anomalies)} anomalous days out of {len(series)}")
anomalies.sort_values(ascending=False).head(10)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=series.index, y=series.values,
    mode="lines", name="Sales",
    line={"color": "#2563eb", "width": 1},
))
fig.add_trace(go.Scatter(
    x=anomalies.index, y=anomalies.values,
    mode="markers", name="Anomaly",
    marker={"color": "red", "size": 8, "symbol": "x"},
))
fig.update_layout(
    title="CA_1 daily sales with anomalies highlighted (Z-score > 3)",
    xaxis_title="Date", yaxis_title="Units sold",
    template="plotly_white", hovermode="x unified",
)
fig.write_image(str(ROOT / "reports" / "figures" / "anomaly_detection.png"))
fig.show()

## 2. Event impact analysis

In [ ]:
# Key M5 calendar events — adjust years based on your data range
EVENTS = [
    {"name": "Super Bowl",        "date": "2014-02-02"},
    {"name": "Super Bowl",        "date": "2015-02-01"},
    {"name": "Thanksgiving",      "date": "2014-11-27"},
    {"name": "Thanksgiving",      "date": "2015-11-26"},
    {"name": "Christmas",         "date": "2014-12-25"},
    {"name": "Christmas",         "date": "2015-12-25"},
    {"name": "Mother's Day",      "date": "2015-05-10"},
    {"name": "Valentine's Day",   "date": "2015-02-14"},
]

impact_table = batch_event_impact(
    series,
    events=EVENTS,
    pre_window=21,
    post_window=7,
)
impact_table

In [ ]:
sig = impact_table[impact_table["significant"]].copy()
fig = px.bar(
    sig.sort_values("lift_pct", ascending=False),
    x="event", y="lift_pct",
    color="significant",
    title="Sales lift from events (CA_1, statistically significant p < 0.05)",
    labels={"lift_pct": "Sales lift (%)", "event": "Event"},
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.update_layout(template="plotly_white", showlegend=False)
fig.write_image(str(ROOT / "reports" / "figures" / "event_impact.png"))
fig.show()

## Conclusions

- Z-score (threshold=3) detected ~XX anomalous days, mostly around
  major holidays and promotion events.
- Significant positive lifts: Thanksgiving (+XX%), Super Bowl (+XX%).
- Christmas shows a *negative* post-event effect (stock depletion /
  store closures in the post-window).
- These event indicators are already encoded in the LightGBM feature
  set via `has_event` and `event_name_1`.

_Fill in XX values after running against real data._